## import Libraries

In [1]:
# !pip install shap

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
import optuna
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)
import numpy as np
import shap


In [3]:
import os
import sys
sys.path.append(os.path.abspath(".."))

In [4]:
from src.Data_preprocessing import preprocess
from src.Feature_Engineering import feature_Engineering
from src.best_preprocessor import  get_best_preprocessor
df=preprocess()
df= feature_Engineering(df)

c:\Users\za813\anaconda3\envs\medical_cost\python.exe
2.2.3


## Hyperparameter Tunning

- **R² : 0.8233 before tunning [with cross validation]**

In [5]:
x=df.drop("charges",axis=1)
y=df["charges"]
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

In [6]:
preprocessor=get_best_preprocessor()

In [7]:
cv = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42 #Locks in the results
)
def objective(trial):

    n_estimators = trial.suggest_int("n_estimators", 100, 500)

    max_depth = trial.suggest_int("max_depth", 5, 30)

    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)

    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="r2"
    )

    return scores.mean()

In [8]:
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)
study.optimize(
    objective,
    n_trials=50
)

[I 2026-08-12 02:04:42,759] A new study created in memory with name: no-name-724a3cff-6418-4893-ba2f-98621b435d96
[I 2026-08-12 02:04:47,539] Trial 0 finished with value: 0.8492164000857079 and parameters: {'n_estimators': 250, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 6}. Best is trial 0 with value: 0.8492164000857079.
[I 2026-08-12 02:04:50,520] Trial 1 finished with value: 0.8521923080977724 and parameters: {'n_estimators': 162, 'max_depth': 9, 'min_samples_split': 3, 'min_samples_leaf': 9}. Best is trial 1 with value: 0.8521923080977724.
[I 2026-08-12 02:04:56,464] Trial 2 finished with value: 0.8520380043521232 and parameters: {'n_estimators': 341, 'max_depth': 23, 'min_samples_split': 2, 'min_samples_leaf': 10}. Best is trial 1 with value: 0.8521923080977724.
[I 2026-08-12 02:05:06,443] Trial 3 finished with value: 0.8390229108382805 and parameters: {'n_estimators': 433, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2}. Best is trial 1 with valu

In [9]:
print(study.best_params)

print(study.best_value) #score r2

print(study.best_trial)

{'n_estimators': 273, 'max_depth': 5, 'min_samples_split': 12, 'min_samples_leaf': 9}
0.85456053149326
FrozenTrial(number=47, state=<TrialState.COMPLETE: 1>, values=[0.85456053149326], datetime_start=datetime.datetime(2026, 8, 12, 2, 8, 37, 1209), datetime_complete=datetime.datetime(2026, 8, 12, 2, 8, 41, 445892), params={'n_estimators': 273, 'max_depth': 5, 'min_samples_split': 12, 'min_samples_leaf': 9}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=500, log=False, low=100, step=1), 'max_depth': IntDistribution(high=30, log=False, low=5, step=1), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=10, log=False, low=1, step=1)}, trial_id=47, value=None)


- **optuna improved R2**

## Train Model

In [10]:
best_model = RandomForestRegressor(
    **study.best_params,
    random_state=42
)

In [11]:
final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_model)
])

## Transformation to target

In [12]:
final_pipeline.fit(X_train, y_train) 

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('one_hot_encoding', ...), ('passthrough', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [13]:
y_pred = final_pipeline.predict(X_test)

## Evaluation

In [14]:
r2 = r2_score(y_test, y_pred)

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

In [15]:
print(f"R² Score : {r2:.4f}")
print(f"MAE      : {mae:.2f}")
print(f"MSE      : {mse:.2f}")
print(f"RMSE     : {rmse:.2f}")


R² Score : 0.8782
MAE      : 2486.82
MSE      : 18908718.45
RMSE     : 4348.42


## Save Model

In [16]:
import joblib

joblib.dump(
    final_pipeline,
    "../models/final_pipeline.pkl"
)

['../models/final_pipeline.pkl']

In [17]:
# loaded_model = joblib.load(
#     "../models/final_pipeline.pkl"
# )

# prediction = loaded_model.predict(X_test)

## Conclusion

**A Random Forest Regressor combined with a preprocessing pipeline achieved the best performance for predicting medical insurance charges. Hyperparameter optimization using Optuna further improved the model's generalization ability. The final model achieved an R² score of 0.878, indicating that it explains approximately 87.8% of the variance in insurance charges. SHAP and Feature Importance analyses showed that smoking status, BMI, and age are the most influential factors affecting insurance costs, while region, sex, and children have relatively minor contributions**